In [2]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils import data
from torchvision import datasets,transforms, utils

USE_MPS = torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

In [2]:
tr_ds = datasets.FashionMNIST(root='../../data',
                              train=True,
                              download=False,
                              transform=transforms.Compose([transforms.ToTensor()]))

In [3]:
BATCH_SIZE=60000
tr_ds_loader=torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)
img,_=next(iter(tr_ds_loader))

img.mean(),img.std()

(tensor(0.2860), tensor(0.3530))

In [4]:
BATCH_SIZE=64
EPOCHS=10

In [5]:
# 데이터 수정 (노이즈 삽입)
# 1. 데이터 준비
transform=transforms.Compose([
    #transforms.RandomHorizontalFlip(),#데이터 증강(노이즈삽입)
    transforms.ToTensor(),#입력 데이터 정리
    transforms.Normalize((0.2860,),(0.3530,))
])#데이터 사용 방식 내용 결정
tr_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('../../data/',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('../../data/',
        train=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [6]:
# 28*28*1
# -> 24*24*10
# -> 12*12*10
#   -> 8*8*20
#   -> 4*4*20 -> 320
#   -> 12*12*20
#   -> 6*6*20 -> 720
# -> 28*28*10
# -> 14*14*10
#   -> 10*10*20
#   -> 5*5*20 -> 500
#   -> 14*14*20
#   -> 7*7*20 -> 980

In [7]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, 5)
        self.conv2 = nn.Conv2d(10, 20, 5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)
    def forward(self, x):
        x = F.max_pool2d(self.conv1(x),2)
        x = F.max_pool2d(self.conv2_drop(self.conv2(x)),2)
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return x

In [8]:
model = Model().to(DEVICE)
opt = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
scheduler = optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.1) # 가급적 epoch 당 결정하라! -> 스케쥴러, 학습이 일어난 다음에 진행되어야 한다.

In [9]:
def train(model, tr_ds_loader, opt, epoch):
    model.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        py = model(data)
        loss = F.cross_entropy(py, target)
        loss.backward()
        opt.step()
        if i%100==0:
            print(f"train epoch {epoch} {i*len(data)}/{len(tr_ds_loader.dataset)}({i*100/len(tr_ds_loader):.3f}%) loss:{loss.item():.3f}")

In [10]:
@torch.no_grad() #서식
def evaluate(model, tt_ds_loader):
    model.eval()
    test_loss = 0
    correct = 0
    for data, target in tt_ds_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        py = model(data)
        test_loss += F.cross_entropy(py, target, reduction='sum').item()
        pred = py.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

In [11]:
for i in range(1, EPOCHS+1):
    train(model, tr_ds_loader, opt, i)
    scheduler.step()
    test_loss, test_accuracy = evaluate(model, tt_ds_loader)

    print(f'epoch: {i}, test_loss: {test_loss:.3f}, test_accuracy: {test_accuracy:.3f}, lr_scheduler: {scheduler.get_last_lr()[0]}')

train epoch 1 0/60000(0.000%) loss:2.324
train epoch 1 6400/60000(10.661%) loss:1.823
train epoch 1 12800/60000(21.322%) loss:1.356
train epoch 1 19200/60000(31.983%) loss:0.891
train epoch 1 25600/60000(42.644%) loss:1.064
train epoch 1 32000/60000(53.305%) loss:0.819
train epoch 1 38400/60000(63.966%) loss:0.768
train epoch 1 44800/60000(74.627%) loss:0.958
train epoch 1 51200/60000(85.288%) loss:0.763
train epoch 1 57600/60000(95.949%) loss:0.750
epoch: 1, test_loss: 0.627, test_accuracy: 75.570, lr_scheduler: 0.01
train epoch 2 0/60000(0.000%) loss:0.742
train epoch 2 6400/60000(10.661%) loss:0.698
train epoch 2 12800/60000(21.322%) loss:0.732
train epoch 2 19200/60000(31.983%) loss:0.833
train epoch 2 25600/60000(42.644%) loss:0.659
train epoch 2 32000/60000(53.305%) loss:0.690
train epoch 2 38400/60000(63.966%) loss:0.492
train epoch 2 44800/60000(74.627%) loss:1.049
train epoch 2 51200/60000(85.288%) loss:0.680
train epoch 2 57600/60000(95.949%) loss:0.522
epoch: 2, test_loss: 0

In [3]:
ck_tr = transforms.Compose([transforms.ToTensor()])
tr_ds = datasets.CIFAR10(root='../../data/', train=True, transform=ck_tr, download=True)
tr_ds_loader = torch.utils.data.DataLoader(tr_ds, batch_size=50000, shuffle=False)

100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s] 


In [13]:
ck_data = iter(tr_ds_loader)
data, _ = next(ck_data)
data

tensor([[[[0.2314, 0.1686, 0.1961,  ..., 0.6196, 0.5961, 0.5804],
          [0.0627, 0.0000, 0.0706,  ..., 0.4824, 0.4667, 0.4784],
          [0.0980, 0.0627, 0.1922,  ..., 0.4627, 0.4706, 0.4275],
          ...,
          [0.8157, 0.7882, 0.7765,  ..., 0.6275, 0.2196, 0.2078],
          [0.7059, 0.6784, 0.7294,  ..., 0.7216, 0.3804, 0.3255],
          [0.6941, 0.6588, 0.7020,  ..., 0.8471, 0.5922, 0.4824]],

         [[0.2431, 0.1804, 0.1882,  ..., 0.5176, 0.4902, 0.4863],
          [0.0784, 0.0000, 0.0314,  ..., 0.3451, 0.3255, 0.3412],
          [0.0941, 0.0275, 0.1059,  ..., 0.3294, 0.3294, 0.2863],
          ...,
          [0.6667, 0.6000, 0.6314,  ..., 0.5216, 0.1216, 0.1333],
          [0.5451, 0.4824, 0.5647,  ..., 0.5804, 0.2431, 0.2078],
          [0.5647, 0.5059, 0.5569,  ..., 0.7216, 0.4627, 0.3608]],

         [[0.2471, 0.1765, 0.1686,  ..., 0.4235, 0.4000, 0.4039],
          [0.0784, 0.0000, 0.0000,  ..., 0.2157, 0.1961, 0.2235],
          [0.0824, 0.0000, 0.0314,  ..., 0

In [14]:
data.shape

torch.Size([50000, 3, 32, 32])

In [15]:
data.mean(dim=[0,2,3])

tensor([0.4914, 0.4822, 0.4465])

In [16]:
data.std(dim=[0,2,3])

tensor([0.2470, 0.2435, 0.2616])

In [17]:
# [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]

In [18]:
BATCH_SIZE=64
EPOCHS=10

In [19]:
# 데이터 수정 (노이즈 삽입)
# 1. 데이터 준비
transform=transforms.Compose([
    #transforms.RandomHorizontalFlip(),#데이터 증강(노이즈삽입)
    transforms.ToTensor(),#입력 데이터 정리
    transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2467, 0.2429, 0.2616))
])#데이터 사용 방식 내용 결정
tr_ds_loader=torch.utils.data.DataLoader(
    datasets.CIFAR10('../../data/',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader=torch.utils.data.DataLoader(
    datasets.CIFAR10('../../data/',
        train=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [20]:
# 모델 설계(res모듈 설계 -> 얕은층)
class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_planes)

        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_planes)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ReModel(nn.Module):
    def __init__(self, class_n):
        super().__init__()
        self.in_plans = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.l1 = self._make_l(16, 2, 1)
        self.l2 = self._make_l(32, 2, 2)
        self.l3 = self._make_l(64, 2, 2)
        self.out_l = nn.Linear(64, class_n)

    def _make_l(self, out_planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_plans, out_planes, stride))
            self.in_plans = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))

        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)

        x = F.avg_pool2d(x, 8)
        x = x.view(x.size(0), -1)
        out = self.out_l(x)
        return out


In [21]:
model = ReModel(10).to(DEVICE)
opt = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0005)
scheduler = optim.lr_scheduler.StepLR(optimizer=opt, step_size=3, gamma=0.1)

In [22]:
model

ReModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (l1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=Tru

In [23]:
def train(model, tr_ds_loader, opt, epoch):
    model.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        py = model(data)
        loss = F.cross_entropy(py, target)
        loss.backward()
        opt.step()

        if i%100==0:
            print(f"train epoch {epoch} {i*len(data)}/{len(tr_ds_loader.dataset)}({i*100/len(tr_ds_loader):.3f}%) loss:{loss.item():.3f}")

In [24]:
@torch.no_grad() #서식
def evaluate(model, tt_ds_loader):
    model.eval()
    test_loss = 0
    correct = 0
    for data, target in tt_ds_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        py = model(data)
        test_loss += F.cross_entropy(py, target, reduction='sum').item()
        pred = py.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

In [25]:
for i in range(1, EPOCHS+1):
    train(model, tr_ds_loader, opt, i)
    scheduler.step()
    test_loss, test_accuracy = evaluate(model, tt_ds_loader)

    print(f'epoch: {i}, test_loss: {test_loss:.3f}, test_accuracy: {test_accuracy:.3f}, lr_scheduler: {scheduler.get_last_lr()[0]}')

train epoch 1 0/50000(0.000%) loss:2.346
train epoch 1 6400/50000(12.788%) loss:1.934
train epoch 1 12800/50000(25.575%) loss:1.680
train epoch 1 19200/50000(38.363%) loss:1.658
train epoch 1 25600/50000(51.151%) loss:1.511
train epoch 1 32000/50000(63.939%) loss:1.161
train epoch 1 38400/50000(76.726%) loss:1.219
train epoch 1 44800/50000(89.514%) loss:1.161
epoch: 1, test_loss: 1.252, test_accuracy: 56.260, lr_scheduler: 0.1
train epoch 2 0/50000(0.000%) loss:0.964
train epoch 2 6400/50000(12.788%) loss:1.139
train epoch 2 12800/50000(25.575%) loss:1.032
train epoch 2 19200/50000(38.363%) loss:1.048
train epoch 2 25600/50000(51.151%) loss:1.029
train epoch 2 32000/50000(63.939%) loss:1.062
train epoch 2 38400/50000(76.726%) loss:0.890
train epoch 2 44800/50000(89.514%) loss:0.920
epoch: 2, test_loss: 1.086, test_accuracy: 63.270, lr_scheduler: 0.1
train epoch 3 0/50000(0.000%) loss:0.810
train epoch 3 6400/50000(12.788%) loss:0.757
train epoch 3 12800/50000(25.575%) loss:0.684
train 